# Search-03c (Rust) — Recherche à écart limité (Limited Discrepancy Search)

> **Partie 3 — Recherche avancée.** Ce notebook prolonge la Partie 3 : techniques
> de recherche heuristique *avancées*, au-delà des fondations de la
> [Partie 1](../Part1-Foundations/Search-03-Informed.ipynb) (A\*, IDA\*, heuristiques
> admissibles). Là où [Search-03b](Search-03b-PatternDatabases.ipynb) fabriquait une
> heuristique *assez informée* pour résoudre à l'optimum par précalcul, ici nous
> partons d'une heuristique *déjà bonne mais imparfaite* et demandons : comment
> trouver l'optimum **sans explorer tout l'arbre** ?

## 1. L'idée : une bonne heuristique se trompe peu

La recherche exhaustive (DFS sur tout l'arbre) trouve l'optimum mais coûte $2^n$
feuilles — prohibitif. La stratégie *greedy* (suivre aveuglément la recommandation
de l'heuristique) est instantanée mais **myope** : elle se contente du premier
chemin, parfois sous-optimal. Entre ces deux extrêmes, **Limited Discrepancy Search**
(Harvey & Ginsberg, 1995) exploite une intuition forte :

> Si l'heuristique est *bonne*, la solution optimale s'écarte de **peu** de ses
> recommandations.

Un **écart** (*discrepancy*) est, à un nœud, le choix d'un enfant que l'heuristique
ne recommande **pas** en premier (rang > 0). LDS(k) énumère exactement les chemins
comportant **au plus k écarts** : à k=0 c'est le greedy ; à k croissant on élargit
progressivement le voisinage du chemin greedy, sans jamais payer le coût exponentiel
de la DFS complète. C'est l'incarnation algorithmique du principe « l'optimum est un
voisin du greedy ». 

## 2. Le problème-banc d'essai : le sac à dos 0/1

Le **sac à dos binaire** (*0/1 knapsack*) est NP-diff et se prête naturellement à
une recherche arborescente. On dispose de $n$ objets, chacun avec un poids $w_i$ et
une valeur $v_i$, et d'une capacité $C$. On veut maximiser la valeur totale sans
dépasser $C$ — en décidant, pour chaque objet, de le **prendre** ou de le **laisser**.
C'est un arbre de décision binaire de profondeur $n$.

L'**heuristique gloutonne** canonique ordonne les objets par **ratio valeur/poids
décroissant** et, à chaque nœud, recommande de **prendre** l'objet courant s'il
rentre. Cette heuristique est *bonne* (les objets à fort ratio sont rarement un
mauvais choix) mais **imparfaite** : un objet à ratio élevé peut en *bloquer* deux
de ratio moyen dont la somme dépasse sa valeur — auquel cas le greedy se trompe.

In [2]:
// Le sac a pos 0/1 : liste d'objets (nom, poids, valeur), ordonnes par RATIO valeur/poids decroissant
// (c'est l'ordre ou l'heuristique gloutonne les considere).
let items: Vec<(&str, i64, i64)> = vec![
    ("A", 6, 8),   // ratio 1.33  <-- top ratio, MAIS poids 6 bloque la paire B+C (valeur 12 > 10)
    ("B", 5, 6),   // ratio 1.20
    ("C", 5, 6),   // ratio 1.20
    ("D", 4, 2),   // ratio 0.50
    ("E", 3, 1),   // ratio 0.33
    ("F", 2, 0),   // ratio 0.00  (objet de valeur nulle -- piege pour le greedy s'il reste de la place)
    ("G", 1, 0),   // ratio 0.00  (objet de valeur nulle -- piege pour le greedy)
];
let capacity: i64 = 10;

// Verifions l'ordre par ratio (l'heuristique suppose cet ordre).
println!("Objets dans l'ordre heuristique (ratio v/w decroissant) :");
for &(name, w, v) in &items {
    let r = if w > 0 { v as f64 / w as f64 } else { f64::INFINITY };
    println!("  {}: ratio = {:.3}", name, r);
}
println!("Capacite C = {}", capacity);

Objets dans l'ordre heuristique (ratio v/w decroissant) :


  A: ratio = 1.333


  B: ratio = 1.200


  C: ratio = 1.200


  D: ratio = 0.500


  E: ratio = 0.333


  F: ratio = 0.000


  G: ratio = 0.000


Capacite C = 10


### Lecture du résultat

La sortie confirme l'ordre heuristique : A en tête avec un ratio de 1.333, suivi de B et C (1.200), puis D (0.500), E (0.333) et F, G (0.000), pour une capacité C = 10. C'est dans cet ordre que le greedy va considérer les objets — et A, en tête de liste, est précisément celui qui va le faire échouer.

In [3]:
// StratEGIE GLOUTONNE (k=0) : suivre l'heuristique aveuglement -- prendre chaque objet s'il rentre.
fn py_list(v: &[String]) -> String {
    format!("[{}]", v.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

fn greedy_knapsack(items: &[(&str, i64, i64)], capacity: i64) -> (i64, Vec<String>) {
    let mut selection: Vec<String> = Vec::new();
    let mut total_w: i64 = 0;
    let mut total_v: i64 = 0;
    for &(name, w, v) in items {
        if total_w + w <= capacity {  // heuristique : prendre si ca rentre
            selection.push(name.to_string());
            total_w += w;
            total_v += v;
        }
    }
    (total_v, selection)
}

let items: Vec<(&str, i64, i64)> = vec![
    ("A", 6, 8),
    ("B", 5, 6),
    ("C", 5, 6),
    ("D", 4, 2),
    ("E", 3, 1),
    ("F", 2, 0),
    ("G", 1, 0),
];
let capacity: i64 = 10;

let (gv, gsel) = greedy_knapsack(&items, capacity);
println!("Greedy (k=0) : valeur = {}, selection = {}", gv, py_list(&gsel));
println!("--> Le greedy prend A (poids 6, valeur 8), puis il ne reste que 4 de capacite :");
println!("    seul D (poids 4) y rentre. Greedy = {{A, D}} = 10. La paire B+C (valeur 12) est bloquee par A.");

Greedy (k=0) : valeur = 10, selection = ['A', 'D']


--> Le greedy prend A (poids 6, valeur 8), puis il ne reste que 4 de capacite :


    seul D (poids 4) y rentre. Greedy = {A, D} = 10. La paire B+C (valeur 12) est bloquee par A.


### Lecture du résultat

La sortie montre le greedy à k=0 : valeur = 10 avec la sélection ['A', 'D']. A (poids 6, valeur 8) occupe 6 des 10 unités de capacité : il ne reste que 4 unités, où seul D (poids 4) rentre — la paire B+C, de valeur 12, est bloquée.

## 3. Le concept d'écart, et l'algorithme LDS(k)

À chaque nœud, l'heuristique **recommande** une action (prendre l'objet s'il rentre,
sinon le laisser). Choisir l'**autre** action coûte **un écart** :

- Objet prenable : rang 0 = **prendre** (recommande), rang 1 = **laisser** (1 écart).
- Objet non prenable (trop lourd) : rang 0 = **laisser** (seule option realisable).

`LDS(k)` descend en privilégiant l'action recommandée, et n'emprunte une branche
« contraire » que si le budget d'écarts `k` n'est pas épuisé. Elle énumère ainsi tous
les chemins à **au plus k** écarts, et retient la meilleure valeur rencontrée. 

In [4]:
// LDS(k_max) sur le sac a pos 0/1 : recursion avec budget d'ecarts
// items : liste (nom, poids, valeur), ordre heuristique (meilleur ratio d'abord).
// Renvoie (meilleure valeur, meilleure selection, nombre de noeuds explores).
fn py_list(v: &[String]) -> String {
    format!("[{}]", v.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

fn lds_knapsack(items: &[(&str, i64, i64)], capacity: i64, k_max: usize) -> (i64, Vec<String>, usize) {
    let n = items.len();
    let mut best_value: i64 = -1;
    let mut best_sel: Vec<String> = Vec::new();
    let mut nodes: usize = 0;

    fn rec(items: &[(&str, i64, i64)], i: usize, rem_cap: i64, val_so_far: i64,
           sel: &mut Vec<String>, disc: usize, k_max: usize,
           best_value: &mut i64, best_sel: &mut Vec<String>, nodes: &mut usize) {
        *nodes += 1;  // feuille : on enregistre une solution
        if i == items.len() {
            if val_so_far > *best_value {
                *best_value = val_so_far;
                *best_sel = sel.clone();
            }
            return;
        }
        let (name, w, v) = items[i];
        let can_take = w <= rem_cap;
        if can_take {
            // Rang 0 (recommande) : PRENDRE -- n'epargne aucun ecart.
            sel.push(name.to_string());
            rec(items, i + 1, rem_cap - w, val_so_far + v, sel, disc, k_max, best_value, best_sel, nodes);
            sel.pop();
            // Rang 1 (contraire) : LAISSER -- coute 1 ecart, si budget reste.
            if disc < k_max {
                rec(items, i + 1, rem_cap, val_so_far, sel, disc + 1, k_max, best_value, best_sel, nodes);
            }
        } else {
            // Objet trop lourd : seule action realisable = laisser (rang 0, sans ecart).
            rec(items, i + 1, rem_cap, val_so_far, sel, disc, k_max, best_value, best_sel, nodes);
        }
    }

    let mut sel: Vec<String> = Vec::new();
    rec(items, 0, capacity, 0, &mut sel, 0, k_max, &mut best_value, &mut best_sel, &mut nodes);
    (best_value, best_sel, nodes)
}

let items: Vec<(&str, i64, i64)> = vec![
    ("A", 6, 8),
    ("B", 5, 6),
    ("C", 5, 6),
    ("D", 4, 2),
    ("E", 3, 1),
    ("F", 2, 0),
    ("G", 1, 0),
];
let capacity: i64 = 10;

// Balayage k = 0, 1, 2, ... : a chaque increment on elargit le voisinage du greedy.
println!("k | valeur | selection        | nœuds explores");
println!("--|--------|-----------------|----------------");
for k in 0..=items.len() {
    let (val, sel, nd) = lds_knapsack(&items, capacity, k);
    println!("{} | {:>6} | {:<15} | {}", k, val, py_list(&sel), nd);
};

k | valeur | selection        | nœuds explores


--|--------|-----------------|----------------


0 |     10 | ['A', 'D']      | 8


1 |     12 | ['B', 'C']      | 19


2 |     12 | ['B', 'C']      | 34


3 |     12 | ['B', 'C']      | 52


4 |     12 | ['B', 'C']      | 73


5 |     12 | ['B', 'C']      | 91


6 |     12 | ['B', 'C']      | 98


7 |     12 | ['B', 'C']      | 99


### Lecture du résultat

Dès k=1, la valeur passe de 10 à 12 (sélection ['B', 'C']) pour 19 nœuds explorés, contre 8 à k=0 ; ensuite la valeur stagne à 12 de k=1 à k=7 tandis que le nombre de nœuds continue de croître (34, 52, 73, 91, 98, 99), jusqu'à 99 nœuds à k=7.

## 4. Référence : l'optimum par énumération exhaustive

Pour prouver que LDS atteint bien le **vrai** optimum (et non une borne locale),
on calcule la solution optimale par énumération exhaustive de l'arbre binaire
complet ($2^n$ feuilles) puis par programmation dynamique. LDS devra converger vers
cette valeur à mesure que $k$ croît — idéalement pour un $k$ **petit**, bien avant
l'énumération complète.

In [5]:
// Optimum exact par enumeration exhaustive (2^n) -- reference de verite.
fn py_list(v: &[String]) -> String {
    format!("[{}]", v.iter().map(|s| format!("'{}'", s)).collect::<Vec<_>>().join(", "))
}

fn exhaustive_knapsack(items: &[(&str, i64, i64)], capacity: i64) -> (i64, Vec<String>, usize) {
    let n = items.len();
    let mut best_v: i64 = -1;
    let mut best_sel: Vec<String> = Vec::new();
    let mut nodes: usize = 0;
    for mask in 0..(1u32 << n) {  // 2^n combinaisons
        nodes += 1;
        let w: i64 = (0..n).map(|i| if (mask >> (n - 1 - i)) & 1 == 1 { items[i].1 } else { 0 }).sum();
        let v: i64 = (0..n).map(|i| if (mask >> (n - 1 - i)) & 1 == 1 { items[i].2 } else { 0 }).sum();
        if w <= capacity && v > best_v {
            best_v = v;
            best_sel = (0..n).filter(|&i| (mask >> (n - 1 - i)) & 1 == 1).map(|i| items[i].0.to_string()).collect();
        }
    }
    (best_v, best_sel, nodes)
}

// Optimum exact par programmation dynamique (pseudo-polynomial) -- 2e reference.
fn dp_knapsack(items: &[(&str, i64, i64)], capacity: i64) -> (i64, Vec<String>) {
    let n = items.len();
    // dp[c] = meilleure valeur avec capacite c
    let mut dp = vec![0i64; (capacity + 1) as usize];
    let mut keep = vec![vec![false; (capacity + 1) as usize]; n];
    for i in 0..n {
        let (_, w, v) = items[i];
        for c in (w..=capacity).rev() {
            if dp[(c - w) as usize] + v > dp[c as usize] {
                dp[c as usize] = dp[(c - w) as usize] + v;
                keep[i][c as usize] = true;
            }
        }
    }
    // Reconstruction de la selection
    let mut sel: Vec<String> = Vec::new();
    let mut c = capacity;
    for i in (0..n).rev() {
        if keep[i][c as usize] {
            sel.push(items[i].0.to_string());
            c -= items[i].1;
        }
    }
    sel.reverse();
    (dp[capacity as usize], sel)
}

let items: Vec<(&str, i64, i64)> = vec![
    ("A", 6, 8),
    ("B", 5, 6),
    ("C", 5, 6),
    ("D", 4, 2),
    ("E", 3, 1),
    ("F", 2, 0),
    ("G", 1, 0),
];
let capacity: i64 = 10;

let (opt_v, opt_sel, ex_nodes) = exhaustive_knapsack(&items, capacity);
let (dp_v, dp_sel) = dp_knapsack(&items, capacity);
println!("Optimum (exhaustif, 2^{} = {} feuilles) : valeur = {}, selection = {}", items.len(), 1 << items.len(), opt_v, py_list(&opt_sel));
println!("Optimum (DP)                                         : valeur = {}, selection = {}", dp_v, py_list(&dp_sel));
println!("nœuds explores par l'exhaustif : {}", ex_nodes);

Optimum (exhaustif, 2^7 = 128 feuilles) : valeur = 12, selection = ['B', 'C']


Optimum (DP)                                         : valeur = 12, selection = ['B', 'C']


nœuds explores par l'exhaustif : 128


### Lecture du résultat

Les deux références exactes concordent : l'énumération exhaustive (2^7 = 128 feuilles, 128 nœuds) et la programmation dynamique donnent toutes deux valeur = 12, sélection ['B', 'C'] — l'optimum que LDS avait déjà atteint dès k=1, pour 19 nœuds seulement.

In [6]:
// Tableau comparatif : greedy vs LDS(k) vs exhaustif -- QUALITE de solution et COUT (noeuds).
fn greedy_knapsack(items: &[(&str, i64, i64)], capacity: i64) -> (i64, Vec<String>) {
    let mut selection: Vec<String> = Vec::new();
    let mut total_w: i64 = 0;
    let mut total_v: i64 = 0;
    for &(name, w, v) in items {
        if total_w + w <= capacity {
            selection.push(name.to_string());
            total_w += w;
            total_v += v;
        }
    }
    (total_v, selection)
}

fn lds_knapsack(items: &[(&str, i64, i64)], capacity: i64, k_max: usize) -> (i64, Vec<String>, usize) {
    let mut best_value: i64 = -1;
    let mut best_sel: Vec<String> = Vec::new();
    let mut nodes: usize = 0;

    fn rec(items: &[(&str, i64, i64)], i: usize, rem_cap: i64, val_so_far: i64,
           sel: &mut Vec<String>, disc: usize, k_max: usize,
           best_value: &mut i64, best_sel: &mut Vec<String>, nodes: &mut usize) {
        *nodes += 1;
        if i == items.len() {
            if val_so_far > *best_value {
                *best_value = val_so_far;
                *best_sel = sel.clone();
            }
            return;
        }
        let (name, w, v) = items[i];
        if w <= rem_cap {
            sel.push(name.to_string());
            rec(items, i + 1, rem_cap - w, val_so_far + v, sel, disc, k_max, best_value, best_sel, nodes);
            sel.pop();
            if disc < k_max {
                rec(items, i + 1, rem_cap, val_so_far, sel, disc + 1, k_max, best_value, best_sel, nodes);
            }
        } else {
            rec(items, i + 1, rem_cap, val_so_far, sel, disc, k_max, best_value, best_sel, nodes);
        }
    }

    let mut sel: Vec<String> = Vec::new();
    rec(items, 0, capacity, 0, &mut sel, 0, k_max, &mut best_value, &mut best_sel, &mut nodes);
    (best_value, best_sel, nodes)
}

fn exhaustive_knapsack(items: &[(&str, i64, i64)], capacity: i64) -> (i64, Vec<String>, usize) {
    let n = items.len();
    let mut best_v: i64 = -1;
    let mut best_sel: Vec<String> = Vec::new();
    let mut nodes: usize = 0;
    for mask in 0..(1u32 << n) {
        nodes += 1;
        let w: i64 = (0..n).map(|i| if (mask >> (n - 1 - i)) & 1 == 1 { items[i].1 } else { 0 }).sum();
        let v: i64 = (0..n).map(|i| if (mask >> (n - 1 - i)) & 1 == 1 { items[i].2 } else { 0 }).sum();
        if w <= capacity && v > best_v {
            best_v = v;
            best_sel = (0..n).filter(|&i| (mask >> (n - 1 - i)) & 1 == 1).map(|i| items[i].0.to_string()).collect();
        }
    }
    (best_v, best_sel, nodes)
}

let items: Vec<(&str, i64, i64)> = vec![
    ("A", 6, 8),
    ("B", 5, 6),
    ("C", 5, 6),
    ("D", 4, 2),
    ("E", 3, 1),
    ("F", 2, 0),
    ("G", 1, 0),
];
let capacity: i64 = 10;

let (opt_v, _opt_sel, ex_nodes) = exhaustive_knapsack(&items, capacity);
let ex_total = ex_nodes;

println!("{:<18}{:>8}{:>11}{:>10}{:>14}", "strategie", "valeur", "optimal?", "nœuds", "% exhaustif");
println!("{}", "-".repeat(61));

fn row(label: &str, val: i64, nd: usize, opt_v: i64, ex_total: usize) {
    let opt = if val == opt_v { "OUI" } else { "non" };
    let pct = if ex_total > 0 { 100.0 * nd as f64 / ex_total as f64 } else { 0.0 };
    println!("{:<18}{:>8}{:>11}{:>10}{:>13.1}%", label, val, opt, nd, pct);
}

let (gv, _gsel) = greedy_knapsack(&items, capacity);
row("greedy (k=0)", gv, items.len() + 1, opt_v, ex_total);
for k in 0..=items.len() {
    let (val, _sel, nd) = lds_knapsack(&items, capacity, k);
    row(&format!("LDS(k={})", k), val, nd, opt_v, ex_total);
}
row("exhaustif (ref)", opt_v, ex_total, opt_v, ex_total);

strategie           valeur   optimal?     nœuds   % exhaustif


-------------------------------------------------------------


greedy (k=0)            10        non         8          6.2%


LDS(k=0)                10        non         8          6.2%


LDS(k=1)                12        OUI        19         14.8%


LDS(k=2)                12        OUI        34         26.6%


... (2 outputs left out)

LDS(k=5)                12        OUI        91         71.1%


LDS(k=6)                12        OUI        98         76.6%


LDS(k=7)                12        OUI        99         77.3%


exhaustif (ref)         12        OUI       128        100.0%


### Lecture du résultat

Le **greedy** (k=0) est **sous-optimal** (valeur 10, sélection {A, D}) : il prend A au
ratio le plus élevé (1,33), qui remplit 6 des 10 unités de capacité et ne laisse de la
place que pour D — ratant la paire B+C dont la valeur (12) est supérieure. Dès **k=1**,
LDS explore le chemin « laisser A » et découvre l'optimum {B, C} = 12 — pour **19 nœuds
seulement**, soit 14,8 % des 128 feuilles de l'exhaustif $2^7$.

Au-delà du $k$ suffisant, LDS continue d'explorer mais **ne trouve rien de meilleur**
(l'optimum est déjà atteint, la valeur stagne à 12 de k=1 à k=7) : c'est le signal pour
arrêter. C'est précisément ce qu'exploite **Iterative LDS** (exercice 1) : incrémenter
$k$ jusqu'à ce que la valeur cesse de croître, sans jamais payer le $2^n$ de l'exhaustif.
La leçon générale tient en une phrase : **quand l'heuristique est bonne, l'optimum est un
proche voisin du greedy, et LDS le trouve en bornant les écarts plutôt qu'en énumérant
l'arbre entier**.

## 5. Ponts avec le reste de la série

| Direction | Lien | Relation |
|-----------|------|----------|
| ← Fondations | [Search-3 — Informed](../Part1-Foundations/Search-03-Informed.ipynb) | A\*, heuristiques admissibles — LDS suppose une heuristique *ordonnante* (qui range les enfants), notion voisine |
| ↔ Search-03b | [Search-03b — Pattern Databases](Search-03b-PatternDatabases.ipynb) | Autre voie avancée : *renforcer* l'heuristique (PDB) plutôt que *borner les écarts* (LDS). Deux réponses à « l'heuristique ne suffit pas » |
| → Partie 4 | [Métaheuristiques](../Part4-Metaheuristics/README.md) | LDS garde l'exigence d'optimalité (la trouve pour k assez grand) ; les métaheuristiques l'abandonnent |

**Référence fondatrice** : Harvey, W. D., & Ginsberg, M. L. (1995) — *Limited
Discrepancy Search*, Proceedings of IJCAI. L'article qui formalise le principe
« l'optimum est un voisin du greedy » et l'algorithme LDS(k).

## 6. Exercices

### Exercice 1 : Iterative LDS (ILDS)
Implémentez **ILDS** : exécutez LDS pour k=0, 1, 2, … en vous arrêtant dès que la
valeur **stagne** (deux increments consécutifs sans amélioration) ou que k atteint n.
Comparez le nombre total de nœuds explorés par ILDS à l'énumération exhaustive sur
cette instance.
- **Indice :** le critère d'arrêt naturel est « la valeur n'a pas augmenté au pas k »
  (l'optimum est trouvé). Attention : stagnation ponctuelle ≠ optimum assuré tant que
  k < n — justifiez votre critère d'arrêt.
- **Objectif :** mesurer le gain (ratio des nœuds ILDS / exhaustif).

In [7]:
// Exercice 1 a completer
// Conseil : bouclez k = 0..n, accumulez les nœuds, arretez quand la valeur cesse de croitre.
// Comparez le total accumule a ex_nodes (l'exhaustif calcule ci-dessus).
let total_nodes_ilds: usize = 0;
println!("Exercice 1 a completer");

Exercice 1 a completer


### Exercice 2 : passage à l'échelle — LDS vs exhaustif
Construisez une instance plus grande (par ex. n=18 objets aléatoires, capacité ~50%
du poids total) où le greedy est sous-optimal. Mesurez :
1. Le **plus petit k** pour lequel LDS atteint l'optimum (vérifié via DP).
2. Le **ratio de nœuds** LDS(k\*) / exhaustif ($2^{18} = 262\,144$ feuilles).
- **Indice :** pour n=18 l'exhaustif est encore calculable ( DP ou 2^18) et sert de
  référence. Observez comment le ratio de nœuds se dégrade quand l'heuristique est
  *mauvaise* (ratio aléatoire) vs *bonne* (vraie valeur/poids).

In [8]:
// Exercice 2 a completer
// Conseil : genere n=18 objets avec des (poids, valeur) aleatoires, calcule l'optimum par DP,
// puis balaye k jusqu'a l'atteindre. Affiche le ratio LDS(k*)/2**18.
println!("Exercice 2 a completer");

Exercice 2 a completer


### Exercice 3 : sensibilité à la qualité de l'heuristique
LDS suppose une heuristique *bonne*. Refaites le balayage k=0..n mais en **perturbant
l'ordre** des objets : (a) ordre aléatoire, (b) ordre par valeur absolue décroissante
(au lieu du ratio). Pour chaque heuristique, notez le **k minimum** pour atteindre
l'optimum.
- **Indice :** plus l'heuristique est *mauvaise*, plus l'optimum s'écarte du greedy,
  donc plus le k nécessaire est grand — LDS y perd son avantage. C'est la limite du
  paradigme : LDS n'est efficace que si l'heuristique est *effectivement* bonne.

In [9]:
// Exercice 3 a completer
// Conseil : permutez `items` selon differents ordres, et pour chacun cherchez le k min.
// Perturbation : essayez une permutation deterministe a graine fixe (LCG maison) pour la reproductibilite.
println!("Exercice 3 a completer");

Exercice 3 a completer


## Conclusion

**Limited Discrepancy Search** occupe une niche précise dans le spectre de la
recherche heuristique. À une extrémité, le **greedy** est instantané mais myope ;
à l'autre, la **DFS exhaustive** est exacte mais exponentielle. LDS exploite l'idée
que *si l'heuristique est bonne, l'optimum est un proche voisin du greedy* : en
bornant le nombre d'écarts (choix contraires à la recommandation), elle trouve
l'optimum pour un coût bien inférieur à $2^n$, **sans renoncer à l'optimalité**
(contrairement aux métaheuristiques de la Partie 4).

La limite est symétrique du avantage : LDS n'est efficace que dans la mesure où
l'heuristique est *effectivement* bonne — une heuristique mauvaise repousse l'optimum
loin du greedy, et le $k$ nécessaire devient si grand que l'énumération bornée rejoint
l'exhaustive (exercice 3). C'est un algorithme qui **parie sur la qualité de
l'heuristique**, et qui gagne ce pari précisément quand les Pattern Databases de
[Search-03b](Search-03b-PatternDatabases.ipynb) ont déjà fait leur travail de
*renforcement* heuristique : les deux notebooks sont complémentaires — l'un fabrique
une meilleure heuristique, l'autre exploite au mieux celle dont on dispose.

> **Prochaines étapes** : LDS se généralise en **ILDS** (itération sur k) et se
> combine avec le *beam search* et l'élagage par borne (*branch-and-bound*). Elle
> s'applique à tout arbre de recherche muni d'une heuristique ordonnante :
> ordonnancement, planification, satisfaction de contraintes.

## À retenir

1. **Le greedy est sous-optimal, et c'est mesuré** : il suit le ratio valeur/poids (A en tête, 1,333) et plafonne à **valeur 10** avec la sélection `{A, D}` — A occupe 6 des 10 unités de capacité et n'en laisse que 4, où seul D rentre ; la paire B+C (valeur 12) est bloquée. L'optimum de référence — énumération exhaustive (2⁷ = 128 feuilles) et programmation dynamique, qui concordent — vaut **12** avec `{B, C}`.
2. **Un seul écart suffit** : LDS(k=1), c'est-à-dire la première branche qui *laisse* A au lieu de le prendre, atteint l'optimum (12, `{B, C}`) pour **19 nœuds**, contre 8 nœuds au k=0, soit **14,8 %** du compte de l'exhaustif.
3. **Au-delà du k utile, on paie pour rien** : la valeur stagne à 12 de k=1 à k=7, pendant que les nœuds continuent de croître (19 → 34 → 52 → 73 → 91 → 98 → 99). Les 80 nœuds supplémentaires entre k=1 et k=7 n'ont produit aucune amélioration de qualité.
4. **Borner les écarts coûte bien moins qu'énumérer l'arbre** : sur cette instance, le compte de nœuds de LDS reste sous celui de l'exhaustif jusqu'à k=7 (99 contre 128, 77,3 % dans le tableau comparatif) ; l'écart ne se creuse pas en exponentiel, parce qu'on ne visite que le voisinage du chemin greedy.
5. **L'algorithme parie sur la qualité de l'heuristique** : un écart suffit parce que le ratio valeur/poids est un bon ordonnancement. La conclusion du carnet en fait la limite symétrique : un ordre perturbé (exercice 3) repousse l'optimum du chemin greedy, et le k nécessaire devient si grand que l'énumération bornée rejoint l'exhaustive.

## Vérifiez votre compréhension

1. **Le greedy prend A, le meilleur ratio, et se trompe.** Que ne tient pas compte le ratio, et pourquoi l'optimum reste-t-il malgré tout proche du chemin greedy ?
2. **De k=0 à k=1, l'optimum est trouvé pour 11 nœuds de plus.** Quelle est l'unique branche empruntée « contre » la recommandation, et pourquoi un seul écart suffit-il ici ?
3. **La valeur stagne de k=1 à k=7.** Un solveur qui itère sur k doit-il continuer jusqu'à k = n ? Sur quoi peut-il s'appuyer pour s'arrêter, et qu'est-ce qu'une stagnation *ne* prouve pas ?

<details><summary>Réponses</summary>

1. Le ratio classe les objets **isolément** : il ne dit rien de la **capacité résiduelle** après un choix lourd. A est le meilleur objet par unité de poids (1,333), mais il consomme 6 des 10 unités — il ne reste que 4 unités, où seul D (valeur 2) rentre, et la paire B+C (poids 10, valeur 12) devient impossible. Le ratio est donc un bon *ordonnancement*, pas une garantie : c'est exactement ce que LDS exploite, puisqu'un seul écart suffit à retrouver l'optimum.

2. La branche est « **laisser A** » (rang 1, donc un écart) ; tout le reste de la descente suit l'ordre heuristique, et cette branche retombe sur le sous-arbre où B puis C rentrent. 19 nœuds = 8 nœuds de la descente greedy + 11 nœuds de la branche « laisser A ». Un seul écart suffit parce que l'erreur du greedy est **locale** : elle tient au choix d'un objet unique, pas à une cascade d'erreurs.

3. Non : continuer jusqu'à k = n revient à payer l'énumération complète (les nœuds montent de 19 à 99 sans que la valeur bouge). On s'appuie sur la **non-décroissance** de la valeur en k : elle ne peut qu'augmenter ou stagner, donc une valeur qui stagne *signale* que le k suivant a peu de chances d'apporter mieux — c'est le critère d'arrêt d'ILDS (exercice 1). Ce que la stagnation **ne prouve pas** : l'optimalité. Ici, seul le contrôle indépendant (exhaustif ou DP, valeur 12) l'établit ; en général, s'arrêter sur une simple stagnation est une heuristique, à justifier par un argument sur le k nécessaire.

</details>

## Pour aller plus loin

- **Jumeau .NET** : [Search-03c-LimitedDiscrepancySearch-CSharp](Search-03c-LimitedDiscrepancySearch-CSharp.ipynb) — la même technique en C#, avec ses propres mesures.
- **Carnet voisin** : [Search-03d-WeightedAstar](Search-03d-WeightedAstar.ipynb) — le **troisième** levier du triptyque : là où LDS *borne les écarts* en conservant l'optimalité, Weighted A\* *relâche* l'optimalité de façon bornée (coût ≤ W × optimal).
- **Dans la même série** : [Search-03b-PatternDatabases](Search-03b-PatternDatabases.ipynb) — le **premier** levier : renforcer l'heuristique par précalcul, au lieu de borner le voisinage du greedy.
- **Exercices du carnet** : l'exercice 1 (ILDS et critère d'arrêt) prolonge le point 3, l'exercice 2 (passage à l'échelle : plus petit k et ratio de nœuds sur n = 18) le point 2, l'exercice 3 (sensibilité à la qualité de l'heuristique) le point 5.
- **Lectures** : Harvey, W. D., & Ginsberg, M. L. (1995), *Limited Discrepancy Search*, Proceedings of IJCAI — la référence fondatrice déjà nommée par le carnet : elle formalise le principe « l'optimum est un proche voisin du greedy ».

> Carnet issu de l'EPIC #26510.